# Project 2 — EDA & Data Cleaning

**AI-Powered Banking Risk & Continuous Audit System**

This notebook is the first gate in the pipeline. It inspects **all raw files before modeling**, profiles quality and distributions, validates table relationships, cleans the data into a separate processed layer, and re-checks integrity after cleaning.

> The dataset is synthetic, designed to have realistic, uneven banking behavior. It is not real customer data.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DIR, PROCESSED_DIR, REPORT_DIR
from src.io_utils import read_raw_tables
from src.eda.profile import build_inventory, build_numeric_profile, build_missing_report, run_eda
from src.cleaning.clean import clean_tables, validate_cleaned_tables

tables = read_raw_tables(RAW_DIR)
print("Project root:", PROJECT_ROOT)
print({name: df.shape for name, df in tables.items()})

## 1. Raw file inventory

Inspect rows, columns, actual CSV file size, in-memory size, missing cells and exact duplicates for every file.

In [ ]:
inventory = build_inventory(tables, RAW_DIR)
display(inventory)

## 2. Column-level profiling

We inspect types, null counts and cardinality before touching the data.

In [ ]:
dtype_rows=[]
for name, df in tables.items():
    for col in df.columns:
        dtype_rows.append({
            "table":name, "column":col, "dtype":str(df[col].dtype),
            "non_null":int(df[col].notna().sum()),
            "nulls":int(df[col].isna().sum()),
            "null_pct":round(df[col].isna().mean()*100,3),
            "unique":int(df[col].nunique(dropna=False))
        })
dtype_report=pd.DataFrame(dtype_rows)
display(dtype_report)

missing=build_missing_report(tables)
if missing.empty:
    print("No missing cells in current raw snapshot.")
else:
    display(missing)

## 3. Duplicate analysis

Exact duplicate rows and duplicate key values are checked separately.

In [ ]:
PRIMARY_KEYS={
    "customers":["customer_id"],
    "accounts":["account_id"],
    "merchants":["merchant_id"],
    "customer_devices":["device_id"],
    "transactions":["transaction_id"]
}
rows=[]
for name,df in tables.items():
    keys=PRIMARY_KEYS.get(name,[])
    rows.append({
        "table":name,
        "exact_duplicate_rows":int(df.duplicated().sum()),
        "duplicate_key_rows":int(df.duplicated(keys,keep=False).sum()) if keys else 0
    })
display(pd.DataFrame(rows))

## 4. Value counts / categorical distributions

Low-cardinality categorical fields are profiled rather than assuming equal frequencies.

In [ ]:
for name,df in tables.items():
    print("\n###",name)
    for col in df.select_dtypes(include=["object"]).columns:
        if col.endswith("_id") or df[col].nunique(dropna=False)>25:
            continue
        vc=df[col].value_counts(dropna=False).rename_axis(col).reset_index(name="count")
        vc["share_pct"]=(vc["count"]/len(df)*100).round(2)
        display(vc)

## 5. Numerical distributions and long-tail checks

Extreme values are flagged for investigation, not automatically deleted. High balances and high transaction values can be legitimate in banking.

In [ ]:
num=build_numeric_profile(tables)
display(num.round(2))

for name,df in tables.items():
    for col in df.select_dtypes(include=np.number).columns:
        if col.endswith("_id"):
            continue
        plt.figure(figsize=(8,4))
        plt.hist(df[col].dropna(),bins=50)
        plt.title(f"{name}: {col}")
        plt.xlabel(col); plt.ylabel("Frequency")
        plt.grid(alpha=.2); plt.tight_layout(); plt.show(); plt.close()

## 6. Date ranges and transaction-time behavior

In [ ]:
for name,df in tables.items():
    for col in df.columns:
        if any(x in col.lower() for x in ["date","time","since","seen"]):
            s=pd.to_datetime(df[col],errors="coerce").dropna()
            if not s.empty:
                print(f"{name}.{col}: {s.min()} → {s.max()}")

tx=tables["transactions"].copy()
tx["transaction_date"]=pd.to_datetime(tx["transaction_date"])
monthly=tx.set_index("transaction_date").resample("MS").size()
monthly.plot(figsize=(10,4),title="Monthly Transaction Volume")
plt.grid(alpha=.2); plt.tight_layout(); plt.show()

## 7. Relational integrity

The model is relational:

`customers → accounts → transactions`

`customers → customer_devices ← transactions`

`merchants ← transactions`

In [ ]:
c,a,m,d,t=[tables[k] for k in ["customers","accounts","merchants","customer_devices","transactions"]]
account_to_customer=a.set_index("account_id")["customer_id"]
opened=a.set_index("account_id")["opening_date"]
checks={
    "accounts→customers":a.customer_id.isin(c.customer_id).all(),
    "transactions→accounts":t.account_id.isin(a.account_id).all(),
    "transactions→merchants":t.merchant_id.isin(m.merchant_id).all(),
    "transactions→devices":t.device_id.isin(d.device_id).all(),
    "transaction customer/account consistency":(t.customer_id==t.account_id.map(account_to_customer)).all(),
    "transaction chronology":(t.transaction_date>=t.account_id.map(opened)).all(),
}
checks

# 8. Clean the raw layer into a processed layer

In [ ]:
cleaned, cleaning_log = clean_tables(tables, PROCESSED_DIR)
display(cleaning_log)

post_clean = validate_cleaned_tables(cleaned)
display(post_clean)
assert post_clean["passed"].all(), "Post-clean validation failed"

## 9. Re-check distributions after cleaning

In [ ]:
for table,col in [("customers","annual_income"),("accounts","balance"),("transactions","amount")]:
    raw_s=pd.to_numeric(tables[table][col],errors="coerce")
    clean_s=pd.to_numeric(cleaned[table][col],errors="coerce")
    print(f"\n{table}.{col}")
    print(pd.DataFrame({"raw":[raw_s.median(),raw_s.quantile(.90),raw_s.quantile(.99),raw_s.max()],
                        "cleaned":[clean_s.median(),clean_s.quantile(.90),clean_s.quantile(.99),clean_s.max()]},
                       index=["median","p90","p99","max"]).round(2))

## 10. Export EDA reports

In [ ]:
run_eda(tables,RAW_DIR,REPORT_DIR/"eda")
print("EDA reports:",REPORT_DIR/"eda")
print("Processed data:",PROCESSED_DIR)

## EDA conclusion

Raw data is preserved, profiling is documented, the processed layer is created separately, and post-clean integrity checks are passed. The next stage is behavioral feature engineering and anomaly modeling.